# പ്രാധാന്യമുള്ള അംഗ മിഡിൽവെയർ ഉപയോഗിച്ച് ഹോട്ടൽ ബുക്കിങ്

ഈ നോട്ട്ബുക്ക് Microsoft ഏജന്റ് ഫ്രെയിംവർക്കിനെ അടിസ്ഥാനമാക്കി **ഫംഗ്ഷൻ-അധിഷ്ഠിത മിഡിൽവെയർ** ആണ് കാണിക്കുന്നത്. നമുക്ക് ഫെഷ്യൽ വർക്ക്‌ഫ്ലോ ഉദാഹരണത്തിൽ നിന്ന് പ്രാഥമിക അംഗങ്ങൾക്ക് പ്രത്യേക ആധികാരങ്ങൾ നൽകുന്ന മിഡിൽവെയർ ലെയർ ചേർക്കുന്നു.

## നിങ്ങൾ പഠിക്കുന്നത്:
1. **ഫംഗ്ഷൻ-അധിഷ്ഠിത മിഡിൽവെയർ**: ഫംഗ്ഷൻ ഫലം തടഞ്ഞ് തിരുത്തൽ
2. **കോൺടെക്സ്റ്റ് ആക്‌സസ്**: നിർവഹണത്തിന് ശേഷം `context.result` വായിക്കുകയും തിരുത്തുകയും ചെയ്യുക
3. **ബിസിനസ് ലജിക് നടപ്പാക്കൽ**: പ്രാധാന്യമുള്ള അംഗ പരമാർത്ഥങ്ങൾ
4. **ഫലം മറിച്ചറിവ്**: ഉപയോക്തൃ നിലയുടെ അടിസ്ഥാനത്തിൽ ഫംഗ്ഷൻ ഫലം മാറ്റുക
5. **അതിർത്തിയില്ലാത്ത വർക്ക്‌ഫ്ലോ, വ്യത്യസ്ത ഫലങ്ങൾ**: മിഡിൽവെയർ-നിർവ്വചിച്ച പെരുമാറ്റ വ്യത്യാസങ്ങൾ

## മിഡിൽവെയർ ഉൾപ്പെട്ട വർക്ക്‌ഫ്ലോ ആർക്കിടെക്ചർ:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## കണ്ടീഷണൽ വർക്ക്‌ഫ്ലോയിൽ നിന്ന് പ്രധാന വ്യത്യാസം:

**മിഡിൽവെയർ ഇല്ലാതെ** (14-conditional-workflow.ipynb):
- പാരിസിൽ റൂമുകൾ ഇല്ല → alternative_agent ൽ റൂട്ടുചെയ്യുക

**മിഡിൽവെയർ ഉപയോഗിച്ച്** (ഈ നോട്ട്ബുക്ക്):
- സാധാരണ ഉപയോക്താവ് + പാരിസ് → റൂമുകൾ ഇല്ല → alternative_agent ൽ റൂട്ടുചെയ്യുക
- പ്രാധാന്യമുള്ള ഉപയോക്താവ് + പാരിസ് → 🌟 മിഡിൽവെയർ മറിച്ചറിവ്! → ലഭ്യമാണ് → booking_agent ൽ റൂട്ടുചെയ്യുക

## ആവശ്യമായ പ്രതിഭാസങ്ങൾ:
- Microsoft ഏജന്റ് ഫ്രെയിംവർക്ക് ഇൻസ്റ്റാൾ ചെയ്തിരിക്കുന്നത്
- കണ്ടീഷണൽ വർക്ക്‌ഫ്ലോകൾ മനസ്സിലാക്കൽ (14-conditional-workflow.ipynb കാണുക)
- GitHub ടോക്കൺ അല്ലെങ്കിൽ OpenAI API കീ
- മിഡിൽവെയർ പാറ്റേണുകൾ അടിസ്ഥാനമായ മനസ്സിലാക്കൽ


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## പടിയ്‌ക്കം 1: ഘടനാപരമായ ഔട്ട്പുട്ടുകൾക്കായി Pydantic മോഡലുകൾ നിർവ്വചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരികെ നൽകുന്ന **സ്കീമ** നിർവ്വചിക്കുന്നു. മിഡിൽവെയർ ലഭ്യതാ ഫലം മാറ്റുമ്പോൾ രേഖപ്പെടുത്തിയിരിക്കാനുള്ള `priority_override` ഫീൽഡ് ഞങ്ങൾ ചേർത്തിട്ടുണ്ട്.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ഘട്ടം 2: പ്രാധാന്യമുള്ള അംഗങ്ങളുടെ ഡാറ്റാബേസ് നിർവചിക്കുക

ഈ ഡെമോയ്ക്കായി, നാം ഒരു പ്രാധാന്യമുള്ള അംഗത്വ ഡാറ്റാബേസ് സിംഗുലേറ്റ് ചെയ്യും. പ്രൊഡക്ഷനിൽ, ഇത് ഒരു യഥാർത്ഥ ഡാറ്റാബേസ് അല്ലെങ്കിൽ API ക്വറി ചെയ്യും.

**പ്രാധാന്യമുള്ള അംഗങ്ങൾ:**
- `alice@example.com` - വൈഐപി അംഗം
- `bob@example.com` - പ്രീമിയം അംഗം  
- `priority_user` - ടെസ്റ്റ് അക്കൗണ്ട്


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ചുവട് 3: ഹോട്ടൽ ബുക്കിംഗ് ടൂൾ സൃഷ്ടിക്കുക

കണ്ടീഷണൽ വർക്‌ഫ്ലോ പോലെ തന്നെയാണിത്, എന്നാൽ ഇപ്പൊഴിതു മിഡിൽവെയർ വഴി തടയപ്പെടും!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ഘട്ടം 4: 🌟 പ്രാധാന്യ പരിശോധന മിഡിൽവെയർ സൃഷ്ടിക്കുക (പ്രധാന ഫീച്ചർ!)

ഇത് ഈ നോട്ടുബുക്കിന്റെ **പ്രഥമ പ്രവർത്തനം** ആണ്. മിഡിൽവെയർ:

1. ഹോട്ടൽ_ബുക്കിംഗ് ഫംഗ്ഷൻ കോൾ **അന്തരീക്ഷം സൃഷ്ടിക്കുന്നു**
2. `next(context)` വിളിച്ച് ഫംഗ്ഷൻ സാധാരണയായി **നടത്തുന്നു**
3. `context.result`-ലുള്ള ഫലം **പരിശോധിക്കുന്നു**
4. ഉപയോക്താവ് പ്രാധാന്യമുള്ളവനും മുറികൾ ലഭ്യമല്ലാത്തപക്ഷം ഫലം **മാറ്റിത്തുല്യപ്പെടുത്തുന്നു**
5. മാറ്റിയ ഫലം ഏജന്റി തിരികെ **നൽകുന്നു**

**പ്രധാന പാറ്റേൺ:**
```python
async def my_middleware(context, next):
    await next(context)  # ഫംഗ്ഷൻ പ്രവർത്തിപ്പിക്കുക
    # ഇപ്പോൾ context.result ഫംഗ്ഷന്റെ ഔട്ട്പുട്ട് ഉൾക്കൊള്ളുന്നു
    if some_condition:
        context.result = new_value  # ഒവർറൈഡ്!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## പടി 5: റൂട്ടിംഗിനുള്ള സ്പഷ്ടികരണ ഫംഗ്ഷനുകൾ നിർവചിക്കുക

അനുകൂല പ്രവൃത്തി ക്രമం പോലെ തന്നെയും തന്നെ സ്പഷ്ടികരണ ഫംഗ്ഷനുകൾ - അവ ഘടിത ഔട്ട്പുട്ട് പരിശോധിച്ച് റൂട്ടിംഗ് നിർവചിക്കുന്നു.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## പടി 6: കസ്റ്റം ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

മുൻപെയും അതുപോലെയുള്ള എക്സിക്യൂട്ടർ - അന്തിമ വർക്‌ഫ്ലോ ഔട്ട്പുട്ട് പ്രദർശിപ്പിക്കുന്നു.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## ഘട്ടം 7: പരിസ്ഥിതി ചാരങ്ങൾ ലോഡ് ചെയ്യുക

LLM ക്ലയന്റ് (Microsoft Foundry / Azure OpenAI) ക്രമീകരിക്കുക.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ഘട്ടം 8: മിഡിൽവെയർ ഉപയോഗിച്ച് AI ഏജന്റുകൾ സൃഷ്ടിക്കുക

**പ്രധാന വ്യത്യാസം:** availability_agent സൃഷ്ടിക്കുമ്പോൾ, `middleware` പാരാമീറ്റർ പാസ്സ് ചെയ്യുന്നു!

ഇതാണ് priority_check_middleware ഏജന്റിന്റെ ഫംഗ്ഷൻ വിളിപ്പിക്കൽ പൈപ്പ്‍ലൈനിൽ എങ്ങനെ ഇഞ്ചെക്ട് ചെയ്യുന്നത്.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ഘട്ടം 9: വർക്ക്‌ഫ്ലോ നിർമ്മിക്കൂ

മുമ്പത്തെപോലെ സമാനമായ വർക്ക്‌ഫ്ലോ ഘടന - ലഭ്യത അടിസ്ഥാനമാക്കിയുള്ള സാന്ദ്രീകരിച്ച റൂട്ടിംഗ്.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## ഘട്ടം 10: ടെസ്റ്റ് കേസ് 1 - പാരീസിലുള്ള സാധാരണ ഉപയോക്താവ് (ഒവർറൈഡ് ഇല്ല)

ഒരു സാധാരണ ഉപയോക്താവ് പാരീസ് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → മുറികൾ ഇല്ല → alternative_agent യിലേക്ക് റൂട്ടുചെയ്യുന്നു


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ഘട്ടം 11: പരിശോധനാവിവരം 2 - 🌟 പ്രാധാന്യമുള്ള ഉപയോക്താവ് പാരിസിൽ (ഒവർറൈഡ് സഹിതം!)

ഒരു പ്രാധാന്യമുള്ള അംഗം പാരിസിൽ ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → ആദ്യം മുറികൾ ഇല്ല → 🌟 മിഡിൽവെയർ ഒവർറൈഡ് ചെയ്യുന്നു! → booking_agent ലേക്ക് മാറ്റുന്നു

**ഇതാണ് മിഡിൽവെയറിന്റെ ശക്തിയുടെ മുഖ്യ പ്രകടനം!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ഘട്ടം 12: ടെസ്റ്റ് കേസ് 3 - സ്റ്റോക്ക്ഹോംയിലെ പ്രാധാന്യപെട്ട ഉപയോക്താവ് (ഇപ്പോഴത്തെ ലഭ്യമാണ്)

പ്രാധാന്യപെട്ട ഉപയോക്താവ് സ്റ്റോക്ക്ഹോം ശ്രമിക്കുന്നു → മുറികൾ ലഭ്യമാണ് → ഓവർറൈഡ് വേണ്ട → ബുക്കിങ് ഏജന്റിലേക്ക് വഴിമാറും

ഇത് കാണിക്കുന്നത് മിഡിൽവെയർ ആവശ്യമാണ് എങ്കിൽ മാത്രമേ പ്രവർത്തിക്കുകയുള്ളൂ എന്നതാണ്!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

##主要要点与中间件概念

### ✅ 你学到了什么：

#### **1. 基于函数的中间件模式**

中间件通过一个简单的异步函数拦截函数调用：

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ഫംഗ്ഷൻ നിർവഹണത്തിന് മുമ്പ്
    print("Intercepting...")
    
    # ഫംഗ്ഷൻ നിർവഹിക്കുക
    await next(context)
    
    # ഫംഗ്ഷൻ നിർവഹണത്തിന് ശേഷമുള്ള ഫലം പരിശോധന ചെയ്യുക
    if context.result:
        # ആവശ്യമുണ്ടെങ്കിൽ ഫലം തിരുത്തുക
        context.result = modified_value
```

#### **2. 上下文访问与结果覆写**

- `context.function` - 访问被调用的函数
- `context.arguments` - 读取函数参数
- `context.kwargs` - 访问额外的参数
- `await next(context)` - 执行函数
- `context.result` - 读取/修改函数输出

#### **3. 业务逻辑实现**

我们的中间件实现了优先成员权益：
- **普通用户**：无修改，标准工作流程
- **优先用户**：将“无可用”覆盖为“有可用”
- **条件逻辑**：只在需要时覆写

#### **4. 相同流程，不同结果**

中间件的力量：
- ✅ 工作流程结构无改动
- ✅ 工具函数无改动
- ✅ 条件路由逻辑无改动
- ✅ 仅中间件→不同行为！

### 🚀 真实应用案例：

1. **VIP/高级功能**
   - 为高级用户覆写速率限制
   - 提供优先资源访问
   - 动态解锁高级功能

2. **A/B 测试**
   - 将用户路由到不同实现
   - 与特定用户测试新功能
   - 渐进式功能发布

3. **安全与合规**
   - 审计函数调用
   - 阻止敏感操作
   - 执行业务规则

4. **性能优化**
   - 为特定用户缓存结果
   - 尽可能跳过高昂操作
   - 动态资源分配

5. **错误处理与重试**
   - 优雅捕获和处理错误
   - 实现重试逻辑
   - 回退到备选实现

6. **日志与监控**
   - 跟踪函数执行时间
   - 记录参数和结果
   - 监控使用模式

### 🔑 与装饰器的关键区别：

| 特性 | 装饰器 | 中间件 |
|---------|-----------|------------|
| **范围** | 单一函数 | 代理中的所有函数 |
| **灵活性** | 定义时固定 | 运行时动态 |
| **上下文** | 限制 | 完整代理上下文 |
| **组合** | 多个装饰器 | 中间件管线 |
| **代理感知** | 否 | 是（访问代理状态） |

### 📚 何时使用中间件：

✅ **使用中间件时：**
- 你需要根据用户/会话状态修改行为
- 你想对多个函数应用逻辑
- 你需要访问代理级上下文
- 你正实现横切关注点（日志、认证等）

❌ **不使用中间件时：**
- 简单输入验证（使用 Pydantic）
- 函数特定逻辑（保持在函数内）
- 一次性修改（直接改函数）

### 🎓 高级模式：

```python
# പലയിടത്തും മിഡിൽവെയർ (നിര്വഹണ ക്രമം പ്രധാനമാണ്!)
middleware=[
    logging_middleware,      # ആദ്യം ലോഗുകൾ
    auth_middleware,         # തുടർന്ന് ഓത് പരിശോധിക്കുന്നു
    cache_middleware,        # തുടർന്ന് കാഷെ പരിശോധിക്കുന്നു
    rate_limit_middleware,   # തുടർന്ന് നിരക്കിൽ പരിധി ഏർപ്പെടുത്തുന്നു
    priority_check_middleware  # അവസാനമായി പ്രാധാന്യ പരിശോധന
]

# ശർത്തി അടിസ്ഥാനമാക്കിയുള്ള മിഡിൽവെയർ നടത്തിപ്പ്
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ഫലം മാറ്റം വരുത്തുക
    else:
        # നിർവഹണം മാനസികമായി ഒഴിവാക്കുക
        context.result = cached_value
```

### 🔗 相关概念：

- **代理中间件**：拦截 agent.run() 调用
- **函数中间件**：拦截工具函数调用（我们用的！）
- **中间件管线**：按顺序执行的中间件链
- **上下文传播**：通过中间件链传递状态


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
